# Lab 07 - Cross Validation: Time-Aware Model Validation

This notebook applies **Lab 7 - Cross Validation** while adapting it for hourly time-series data.


## Lab 7 concepts used

- Build a regression pipeline.
- Compare ordinary K-fold validation with `TimeSeriesSplit`.
- Use cross-validation metrics to check whether a model is stable.
- Keep the final assignment's chronological-validation requirement explicit.

For time-series data, shuffled K-fold validation can give optimistic results because future patterns may leak into training folds. `TimeSeriesSplit` is the safer default for final reporting.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'AML Assignment' and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT


In [ ]:
DATASET_FILENAME = 'global_urban_smog_pm25_hourly.csv'
matches = sorted((PROJECT_ROOT / 'Datasets').glob(f'*/{DATASET_FILENAME}'))
if not matches:
    raise FileNotFoundError(f'Could not find {DATASET_FILENAME} under {PROJECT_ROOT / "Datasets"}')
DATASET_PATH = matches[0]
data = pd.read_csv(DATASET_PATH)
data['Timestamp'] = pd.to_datetime(data['Timestamp'])
data = data.sort_values(['Timestamp', 'City']).reset_index(drop=True)
print(DATASET_PATH)
data.head()


In [ ]:
def add_time_features(df):
    out = df.copy()
    out['Timestamp'] = pd.to_datetime(out['Timestamp'])
    out = out.sort_values(['Timestamp', 'City']).reset_index(drop=True)
    out['hour'] = out['Timestamp'].dt.hour
    out['dayofweek'] = out['Timestamp'].dt.dayofweek
    out['month'] = out['Timestamp'].dt.month
    out['dayofyear'] = out['Timestamp'].dt.dayofyear
    out['is_weekend'] = out['dayofweek'].isin([5, 6]).astype(int)
    return out

def chronological_split(df, train_size=0.8):
    split_idx = int(len(df) * train_size)
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()

def latest_rows(df, max_rows):
    if len(df) <= max_rows:
        return df.copy()
    return df.tail(max_rows).copy()

REGRESSION_NUMERIC_NO_AQI = [
    'Latitude', 'Longitude', 'PM10_ug_m3', 'Carbon_Monoxide_ug_m3',
    'Nitrogen_Dioxide_ug_m3', 'Ozone_ug_m3', 'Dust_ug_m3',
    'UV_Index', 'hour', 'dayofweek', 'month', 'is_weekend'
]
CATEGORICAL_FEATURES = ['City']


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_preprocess, REGRESSION_NUMERIC_NO_AQI),
        ('cat', categorical_preprocess, CATEGORICAL_FEATURES),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_validate

model_df = latest_rows(add_time_features(data), 60000)
X = model_df[REGRESSION_NUMERIC_NO_AQI + CATEGORICAL_FEATURES]
y = model_df['PM2_5_ug_m3']

ridge_pipeline = Pipeline(steps=[('preprocess', preprocessor), ('model', Ridge(alpha=1.0))])


In [ ]:
scoring = {'RMSE': 'neg_root_mean_squared_error', 'R2': 'r2'}
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
time_split = TimeSeriesSplit(n_splits=5)

kfold_scores = cross_validate(ridge_pipeline, X, y, cv=kfold, scoring=scoring, n_jobs=-1)
time_scores = cross_validate(ridge_pipeline, X, y, cv=time_split, scoring=scoring, n_jobs=-1)

summary = pd.DataFrame({
    'method': ['Shuffled KFold', 'TimeSeriesSplit'],
    'mean_RMSE': [-kfold_scores['test_RMSE'].mean(), -time_scores['test_RMSE'].mean()],
    'std_RMSE': [kfold_scores['test_RMSE'].std(), time_scores['test_RMSE'].std()],
    'mean_R2': [kfold_scores['test_R2'].mean(), time_scores['test_R2'].mean()],
    'std_R2': [kfold_scores['test_R2'].std(), time_scores['test_R2'].std()],
})
summary


In [ ]:
fold_df = pd.DataFrame(
    [{'fold': i, 'method': 'TimeSeriesSplit', 'RMSE': score}
     for i, score in enumerate(-time_scores['test_RMSE'], start=1)]
    + [{'fold': i, 'method': 'Shuffled KFold', 'RMSE': score}
       for i, score in enumerate(-kfold_scores['test_RMSE'], start=1)]
)
fold_df


In [ ]:
plt.figure(figsize=(7, 4))
for method, group in fold_df.groupby('method'):
    plt.plot(group['fold'], group['RMSE'], marker='o', label=method)
plt.xlabel('Fold')
plt.ylabel('RMSE')
plt.title('Cross-validation RMSE by fold')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## What was learned from Lab 7

Cross-validation needs to respect the data-generating process. For final reporting, use chronological holdout and/or `TimeSeriesSplit`; shuffled K-fold results can still be shown as a cautionary comparison, not as the main evidence.
